# Phase 7 probe sweep — Kaggle CPU (split run: olmoe models only)

Runs the full Phase-7 predictor panel (F0–F6, FV) for the **three OLMoE models**
(olmoe-0125, -instruct, -0924); the four heavier models (gpt-oss, deepseek, qwen3, gemma) are being
run on the local workstation in parallel. Pulls trace streams from HF. The probe fix is baked in:
**F4/FV use the convergent L-BFGS solver**, F2/F3/F6 keep AdamW (already converged), F5 is the AdamW
MLP. No flags needed. Merge this notebook's `phase7_results.zip` with the local `results/probes/`.

**Before running:** CPU-only accelerator; Internet ON; a Kaggle Secret named `HF_TOKEN` with read
access to the trace repos. Upload the code zip (`phase7_kaggle_cpu.zip`) either as a Kaggle Dataset
or straight into the notebook — the next cell finds it.


In [ ]:
!pip -q install -U huggingface_hub

In [ ]:
# --- locate the code (Kaggle auto-extracts uploaded zips; also handles a raw zip) and copy to a writable dir ---
import glob, zipfile, os, shutil
CODE = '/kaggle/working/code'
shutil.rmtree(CODE, ignore_errors=True)
hits = glob.glob('/kaggle/input/**/scripts/phase7_sweep.py', recursive=True)
if hits:
    src = os.path.dirname(os.path.dirname(hits[0]))   # dir containing scripts/, src/, corpora/
    shutil.copytree(src, CODE)
    print("copied extracted code", src, "->", CODE)
else:
    zips = glob.glob('/kaggle/input/**/phase7_kaggle_cpu.zip', recursive=True) + \
           glob.glob('/kaggle/working/*.zip')
    assert zips, "code not found: expected scripts/phase7_sweep.py or phase7_kaggle_cpu.zip under /kaggle/input"
    zipfile.ZipFile(zips[0]).extractall(CODE)
    print("extracted", zips[0], "->", CODE)
for need in ["scripts/phase7_sweep.py", "src/probes/linear.py", "corpora/mixed-v2.splits.json"]:
    assert os.path.exists(os.path.join(CODE, need)), f"missing {need} in {CODE}"
assert 'lbfgs' in open(os.path.join(CODE, 'src/probes/linear.py')).read(), "old code: L-BFGS fix missing"
print("code OK:", sorted(os.listdir(CODE)))

In [ ]:
# --- per-model: download that model's streams (NO logits), sweep, delete; keeps peak disk ~10GB ---
# Phase 7 reads tokens/topk/hidden only, so logits.bin (the bulk of the 94GB) is skipped and the
# sweep runs with --no-validate-sizes. Fully resumable: results persist in /kaggle/working.
import os, sys, shutil, subprocess, time
from huggingface_hub import snapshot_download
from kaggle_secrets import UserSecretsClient
tok = UserSecretsClient().get_secret("HF_TOKEN")
CODE='/kaggle/working/code'; ROOT='/kaggle/temp/traces'
OUT='/kaggle/working/results/probes'; os.makedirs(OUT, exist_ok=True)
env=dict(os.environ, PYTHONPATH=CODE, OMP_NUM_THREADS="4", OPENBLAS_NUM_THREADS="4",
         MKL_NUM_THREADS="4", NUMEXPR_NUM_THREADS="4")
FEATS=["F0","F1","F2","F3","F4","F5","F6","FV"]
ENGINE={"olmoe-0125":"vllm","olmoe-0125-instruct":"vllm","olmoe-0924":"vllm"}  # split: Kaggle = light olmoe models only
REPO={"llamacpp":"Ryze242005/llamacpp-traces-v2","vllm":"Ryze242005/vllm-traces-v2"}
for m,eng in ENGINE.items():
    dst=f"{ROOT}/{eng}"
    print(f"\n=== {m} ({eng}) : download ===", flush=True)
    snapshot_download(REPO[eng], repo_type="dataset", local_dir=dst, token=tok,
                      allow_patterns=[f"traces/{m}/mixed-v2/**"],
                      ignore_patterns=["*logits.bin"])
    print(f"=== {m} : sweep ===", flush=True)
    rc=subprocess.run([sys.executable,"-m","scripts.phase7_sweep","--root",ROOT,"--models",m,
        "--features",*FEATS,"--splits","corpora/mixed-v2.splits.json","--out",OUT,
        "--no-validate-sizes"], cwd=CODE, env=env).returncode
    print(f"=== {m} : rc={rc}; freeing disk ===", flush=True)
    shutil.rmtree(f"{dst}/traces/{m}", ignore_errors=True)
print("\nSWEEP COMPLETE")

In [ ]:
# --- summary + package results ---
import json, glob, os, shutil
from collections import defaultdict
OUT='/kaggle/working/results/probes'
rows=defaultdict(dict); negs=0
for f in glob.glob(OUT+'/*.json'):
    d=json.load(open(f)); m=d['model']; feat=d['feature']
    fit=[L for L in d['layers'].values() if 'metrics' in L]
    if not fit: continue
    r=sum(L['metrics']['family_b']['ratio'] for L in fit)/len(fit)
    a=sum(L['metrics']['family_a']['set_agreement@k'] for L in fit)/len(fit)
    negs+=sum(1 for L in fit if L['metrics']['family_b']['negative'])
    rows[m][feat]=(r,a)
feats=["F0","F1","F2","F3","F4","F5","F6","FV"]
print("mean Î/H (set_agreement@k) by model x feature:\n")
print("model".ljust(22)+"".join(f.rjust(14) for f in feats))
for m in sorted(rows):
    line=m.ljust(22)
    for f in feats:
        line+= (f"{rows[m][f][0]:.2f}/{rows[m][f][1]:.2f}".rjust(14)) if f in rows[m] else "-".rjust(14)
    print(line)
print(f"\nnegative-Î cells: {negs}")
shutil.make_archive('/kaggle/working/phase7_results','zip',OUT)
print("\npackaged -> /kaggle/working/phase7_results.zip  (download this)")